## Picture depicts  
M185339187|P180|Q937


In [1]:
from datetime import datetime
start_time  = datetime.now()
print("Last run: ", start_time)

Last run:  2026-03-07 21:33:55.414611


In [4]:
import requests
import pandas as pd

SPARQL_COMMONS = "https://commons-query.wikimedia.org/sparql"

HEADERS = {
    "User-Agent": "SPA-sync-federated/1.0 (salgo60@msn.com)"
}

query = """
SELECT ?media ?spa ?person WHERE {

  ?media wdt:P4819 ?spa .

  FILTER NOT EXISTS {
    ?media wdt:P180 ?any
  }

  SERVICE <https://query.wikidata.org/sparql> {

    ?person wdt:P2488 ?spa ;
            wdt:P31 wd:Q5 .

  }

}
"""

print("Running federated SPARQL...")

r = requests.get(
    SPARQL_COMMONS,
    params={"query": query, "format": "json"},
    headers=HEADERS,
    timeout=60
)

r.raise_for_status()

data = r.json()

rows = data["results"]["bindings"]

print("Rows:", len(rows))

Running federated SPARQL...


JSONDecodeError: Expecting value: line 1 column 1 (char 0)

In [2]:
import requests
import pandas as pd

SPARQL_COMMONS = "https://query.wikimedia.org/sparql"
SPARQL_WD = "https://query.wikidata.org/sparql"

HEADERS = {
    "User-Agent": "SPA-sync-sparql/1.0 (salgo60@msn.com)"
}

# ------------------------------------------------------------
# 1. Commons: MediaInfo with SPA ID but missing depicts
# ------------------------------------------------------------

query_commons = """
SELECT ?media ?spa WHERE {

  ?media wdt:P4819 ?spa .

  FILTER NOT EXISTS {
    ?media wdt:P180 ?any
  }

}
"""

r = requests.get(
    SPARQL_COMMONS,
    params={"query": query_commons, "format": "json"},
    headers=HEADERS,
)

r.raise_for_status()

commons_data = r.json()
commons_rows = commons_data["results"]["bindings"]

print("Commons rows:", len(commons_rows))


# ------------------------------------------------------------
# 2. Wikidata: SPA ID → person
# ------------------------------------------------------------

query_wd = """
SELECT ?person ?spa WHERE {
  ?person wdt:P31 wd:Q5 ;
          wdt:P2488 ?spa .
}
"""

r = requests.get(
    SPARQL_WD,
    params={"query": query_wd, "format": "json"},
    headers=HEADERS,
)

r.raise_for_status()

wd_data = r.json()

spa_map = {}

for row in wd_data["results"]["bindings"]:
    spa = row["spa"]["value"]
    person = row["person"]["value"].split("/")[-1]
    spa_map[spa] = person

print("SPA persons loaded:", len(spa_map))


# ------------------------------------------------------------
# 3. Join Commons data with Wikidata mapping
# ------------------------------------------------------------

records = []

for row in commons_rows:

    media = row["media"]["value"].split("/")[-1]
    spa = row["spa"]["value"]

    person = spa_map.get(spa)

    if not person:
        continue

    records.append({
        "media": media,
        "spa": spa,
        "person": person
    })

df = pd.DataFrame(records)

print("Matches:", len(df))
display(df.head())


# ------------------------------------------------------------
# 4. Generate QuickStatements
# ------------------------------------------------------------

qs = []

for r in records:

    media = r["media"]
    spa = r["spa"]
    person = r["person"]

    qs.append(f'{media}|P4819|"{spa}"')
    qs.append(f"{media}|P180|{person}")


# ------------------------------------------------------------
# 5. Save file
# ------------------------------------------------------------

outfile = "quickstatements_spa_depicts_sparql.txt"

with open(outfile, "w") as f:
    f.write("\n".join(qs))

print("Statements written:", len(qs))
print("File:", outfile)

ConnectionError: HTTPSConnectionPool(host='query.wikimedia.org', port=443): Max retries exceeded with url: /sparql?query=%0ASELECT+%3Fmedia+%3Fspa+WHERE+%7B%0A%0A++%3Fmedia+wdt%3AP4819+%3Fspa+.%0A%0A++FILTER+NOT+EXISTS+%7B%0A++++%3Fmedia+wdt%3AP180+%3Fany%0A++%7D%0A%0A%7D%0A&format=json (Caused by NameResolutionError("HTTPSConnection(host='query.wikimedia.org', port=443): Failed to resolve 'query.wikimedia.org' ([Errno 8] nodename nor servname provided, or not known)"))

In [ ]:
qs[:10]

In [ ]:
end = datetime.now()
print("Ended: ", end) 
print('Time elapsed (hh:mm:ss.ms) {}'.format(datetime.now() - start_time))